# 01 · Setup, data check and cohort audit

Upload the Aspera download to `MyDrive/bmets/UCSD-BMETS-Longitudinal/` and the four TSVs to `MyDrive/bmets/clinical/`. You need about 350 GB of Drive space (raw data plus the preprocessed cache). Otherwise run preprocessing on Juno and copy `cache/pp` to Drive.

In [ ]:
# --- setup (run every session) ---
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['BMF_DRIVE'] = '/content/drive/MyDrive/bmets'                       # change if you use another folder
os.environ['BMF_DATA'] = os.environ['BMF_DRIVE'] + '/UCSD-BMETS-Longitudinal'   # Aspera download target (uploaded to Drive)
os.environ['SAMMED3D_REPO'] = os.environ['BMF_DRIVE'] + '/weights/SAM-Med3D'     # cloned by scripts/get_weights.sh
os.makedirs(os.environ['BMF_DRIVE'], exist_ok=True)
REPO = '/content/claude_projects'
if not os.path.exists(REPO):
    !git clone -q https://github.com/rittik047/claude_projects.git $REPO
%cd /content/claude_projects/bmets-forecast
!pip -q install -e ".[dl,extra]"
CFG = 'configs/colab.yaml'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pytest -q -p no:warnings tests  # synthetic tests: must pass

In [ ]:
!bmf audit --config $CFG
!bmf index --config $CFG
!bmf splits --config $CFG

In [ ]:
import json; print(json.dumps(json.load(open(os.environ['BMF_DRIVE'] + '/reports/audit.json')), indent=1)[:2000])

**Weights.** Download SAM-Med3D-turbo and the BrainSegFounder Drive folder, then link the BraTS fine-tuned 4-channel checkpoint as `weights/brainsegfounder_swinunetr_brats.pt`.

In [ ]:
!bash scripts/get_weights.sh $BMF_DRIVE/weights
# !ln -sf "$BMF_DRIVE/weights/brainsegfounder_drive/<chosen>.pt" $BMF_DRIVE/weights/brainsegfounder_swinunetr_brats.pt

**Pre-flight check.** This checks the label scheme against MRI contrast physics (ET brightest on t1c, oedema bright on FLAIR, no old-BraTS ET=4) and loads each checkpoint. A wrong scheme or a wrong checkpoint stops here.

In [ ]:
!bmf check --config $CFG
import json; print(json.dumps(json.load(open(os.environ['BMF_DRIVE'] + '/reports/check.json')), indent=1)[:3000])